# Benchmark 1 of 3: load a benchmark into Neo4j

This notebook gets a benchmark ready to run:

1. pick a benchmark and how many questions to use,
2. load the questions' knowledge graphs into Neo4j,
3. check that everything is there.

Then open `2_run_benchmark.ipynb`.

| Benchmark | Questions | Graph | Score |
|-|-|-|-|
| `metaqa` | Template questions about films, 1 to 3 steps | One film knowledge base (43,000 entities) | Hits@1 |
| `webqsp` | Real search-engine questions | A small Freebase subgraph for each question | F1 |
| `cwq` | Harder questions built from WebQSP | A small Freebase subgraph for each question | P@1 |

## Settings

Change these, then run the notebook from top to bottom.

In [ ]:
BENCHMARK = "webqsp"   # "metaqa", "webqsp" or "cwq"
SIZE = 50              # how many questions; the sample in data/samples/ has 200 per benchmark
SOURCE = "sample"      # "sample" uses the questions that ship with the repo; "huggingface" downloads the full benchmark

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))  # so the notebook can import utils/ from the repository root

import json

import pandas as pd
from tqdm.auto import tqdm

from utils import data, graph
from utils.settings import DATA

driver = graph.connect()
graph.setup(driver)
print("Connected to Neo4j.")

## The questions

Each question names the entity to start from, and lists its gold answers.

In [ ]:
questions = data.questions_for(BENCHMARK, size=SIZE, source=SOURCE)
print(f"{len(questions)} questions from {data.BENCHMARKS[BENCHMARK].title}")
pd.DataFrame([{"question": q["question"], "start from": ", ".join(q["start"]), "gold answers": ", ".join(q["answers"][:3])}
              for q in questions[:5]])

## Load the graphs into Neo4j

WebQSP and CWQ give every question its own small subgraph. MetaQA has one film knowledge base that all its questions
share. Graphs that are already loaded are skipped, so running this again is quick.

In [ ]:
loaded = graph.loaded_scopes(driver, BENCHMARK)
if BENCHMARK == "metaqa":
    if "kb" not in loaded:
        kb = data.read_metaqa_kb(None if SOURCE == "sample" else data.download("metaqa"))
        print(f"Loading the film knowledge base: {graph.load_triples(driver, 'metaqa', 'kb', kb):,} facts")
else:
    for q in tqdm([q for q in questions if q["qid"] not in loaded], desc="Loading question graphs"):
        graph.load_triples(driver, BENCHMARK, q["qid"], q["graph"])

scopes = ["kb"] if BENCHMARK == "metaqa" else [q["qid"] for q in questions]
counts = graph.count(driver, BENCHMARK, scopes)
print(f"In Neo4j for these questions: {counts['entities']:,} entities and {counts['facts']:,} facts")

## Look at one question

These are the connections the walk can choose from at its first step. You can also explore the graph in the Neo4j
Browser at http://localhost:7474.

In [ ]:
q = questions[0]
scope = graph.scope_of(BENCHMARK, q)
records, _, _ = driver.execute_query(
    "MATCH (e:KGEntity)-[r:KG_EDGE]-(t) WHERE e.key IN $keys "
    "RETURN r.relation AS relation, count(t) AS targets, collect(t.name)[..3] AS examples ORDER BY targets DESC LIMIT 10",
    keys=[graph.key(BENCHMARK, scope, name) for name in q["start"]])
print("Question:", q["question"])
print("Start from:", ", ".join(q["start"]))
pd.DataFrame([r.data() for r in records])

## Save the questions for the next notebook

`2_run_benchmark.ipynb` runs exactly these questions.

In [ ]:
path = DATA / "loaded" / f"{BENCHMARK}.json"
path.parent.mkdir(parents=True, exist_ok=True)
path.write_text(json.dumps({"benchmark": BENCHMARK, "size": SIZE, "source": SOURCE,
                            "questions": [{k: v for k, v in q.items() if k != "graph"} for q in questions]}))
driver.close()
print(f"Saved {len(questions)} questions to {path.relative_to(DATA.parent)}. Next: 2_run_benchmark.ipynb")